# Difference-in-Differences (DiD) Analysis

## Overview

This notebook estimates the average treatment effect of hospital mergers using a Difference-in-Differences (DiD) framework.

It follows the same analytic data design as the parallel-trends check:

- `treated`: 1 for hospitals that ever experience a merger
- `post_merger`: 1 for treated hospitals in years on/after their first merger year
- Outcome is modeled in logs: `log_total_operating_costs = log(1 + total_operating_costs)`

Standard errors are clustered at the hospital level (`ccn`).

In [1]:
# Imports
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

In [2]:
# Load hospital analysis dataset
cwd = Path.cwd()
project_root = cwd if (cwd / "01_data").exists() else cwd.parent

data_path = project_root / "01_data" / "hospital_analysis.csv"
hospital_analysis = pd.read_csv(data_path)

df = hospital_analysis.copy()

# Standardize key fields used by DiD
df["fiscal_year"] = pd.to_numeric(df["fiscal_year"], errors="coerce")
df["treated"] = pd.to_numeric(df["treated"], errors="coerce")
df["post_merger"] = pd.to_numeric(df["post_merger"], errors="coerce")

# Clean outcome so log(1 + x) is well-defined
df["total_operating_costs"] = pd.to_numeric(df["total_operating_costs"], errors="coerce")
df = df[df["total_operating_costs"].notna()].copy()
df = df[df["total_operating_costs"] >= 0].copy()
df["log_total_operating_costs"] = np.log1p(df["total_operating_costs"])

# Bankruptcy is used as an outcome via a linear probability model (OLS)
if "bankruptcy" in df.columns:
    df["bankruptcy"] = pd.to_numeric(df["bankruptcy"], errors="coerce")

print("Loaded DiD frame shape:", df.shape)
print("Treated hospitals:", df.loc[df["treated"] == 1, "ccn"].nunique())
print("Control hospitals:", df.loc[df["treated"] == 0, "ccn"].nunique())

Loaded DiD frame shape: (53458, 31)
Treated hospitals: 1242
Control hospitals: 5227


In [3]:
# Descriptive plot: average log costs by fiscal year
plot_df = (
    df.groupby(["fiscal_year", "treated"], as_index=False)["log_total_operating_costs"]
    .mean()
)

plot_df["group"] = plot_df["treated"].map({0: "Never treated", 1: "Treated"})

plt.figure(figsize=(9, 5))
sns.lineplot(
    data=plot_df,
    x="fiscal_year",
    y="log_total_operating_costs",
    hue="group",
    marker="o",
)

plt.title("Average Log Operating Costs by Fiscal Year")
plt.xlabel("Fiscal year")
plt.ylabel("Mean log(1 + total operating costs)")
plt.xticks(sorted(df["fiscal_year"].dropna().unique().tolist()))
plt.legend()
plt.tight_layout()
# Save instead of showing to keep nbconvert output small in headless runs.
out_dir = project_root / "05_outputs" / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / "did_avg_log_costs_by_fiscal_year.png", dpi=200)
plt.close()

In [4]:
# Main DiD regression for operating costs
# Specification:
# The coefficient on (treated:post_merger) is the DiD estimate.

reg_df = df.dropna(
    subset=["fiscal_year", "treated", "post_merger", "log_total_operating_costs"]
).copy()

# Cast to numpy int after NA-drop (patsy prefers non-nullable dtypes).
for _col in ["fiscal_year", "treated", "post_merger"]:
    reg_df[_col] = reg_df[_col].astype(int)

cost_model = smf.ols(
    "log_total_operating_costs ~ treated + treated:post_merger + C(fiscal_year)",
    data=reg_df,
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_df["ccn"]},
)

print("DiD (Operating Costs) - clustered by ccn")
print("=" * 70)
print(cost_model.summary())

att_log_points = float(cost_model.params.get("treated:post_merger", np.nan))
if np.isfinite(att_log_points):
    # Approx percent effect in levels: exp(beta) - 1
    att_pct = 100.0 * (np.exp(att_log_points) - 1.0)
    print(f"\nDiD ATT (log points): {att_log_points:.6f}")
    print(f"DiD ATT (approx % change in costs): {att_pct:.2f}%")

DiD (Operating Costs) - clustered by ccn
                                OLS Regression Results                               
Dep. Variable:     log_total_operating_costs   R-squared:                       0.027
Model:                                   OLS   Adj. R-squared:                  0.027
Method:                        Least Squares   F-statistic:                     67.05
Date:                       Thu, 16 Apr 2026   Prob (F-statistic):          1.04e-130
Time:                               09:25:21   Log-Likelihood:                -96685.
No. Observations:                      53458   AIC:                         1.934e+05
Df Residuals:                          53447   BIC:                         1.935e+05
Df Model:                                 10                                         
Covariance Type:                     cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
--------

In [5]:
# DiD regression for bankruptcy (linear probability model)
if "bankruptcy" in reg_df.columns:
    bank_df = reg_df.dropna(subset=["bankruptcy"]).copy()
    
    for _col in ["fiscal_year", "treated", "post_merger"]:
        bank_df[_col] = bank_df[_col].astype(int)

    bank_model = smf.ols(
        "bankruptcy ~ treated + treated:post_merger + C(fiscal_year)",
        data=bank_df,
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": bank_df["ccn"]},
    )

    print("\nDiD (Bankruptcy) - clustered by ccn")
    print("=" * 70)
    print(bank_model.summary())

    att_bank = float(bank_model.params.get("treated:post_merger", np.nan))
    if np.isfinite(att_bank):
        print(f"\nDiD ATT (probability points): {att_bank:.6f}")
else:
    print("Column `bankruptcy` not found; skipping bankruptcy regression.")


DiD (Bankruptcy) - clustered by ccn
                            OLS Regression Results                            
Dep. Variable:             bankruptcy   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.647
Date:                Thu, 16 Apr 2026   Prob (F-statistic):             0.0871
Time:                        09:25:21   Log-Likelihood:             1.3790e+05
No. Observations:               53458   AIC:                        -2.758e+05
Df Residuals:                   53447   BIC:                        -2.757e+05
Df Model:                          10                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------

In [6]:
# Robustness check: interaction-only specification
rob_cost = smf.ols(
    "log_total_operating_costs ~ treated:post_merger + C(fiscal_year)",
    data=reg_df,
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_df["ccn"]},
)

print("\nRobustness: interaction-only (operating costs)")
print("=" * 70)
print(rob_cost.summary())


Robustness: interaction-only (operating costs)
                                OLS Regression Results                               
Dep. Variable:     log_total_operating_costs   R-squared:                       0.011
Model:                                   OLS   Adj. R-squared:                  0.011
Method:                        Least Squares   F-statistic:                     59.58
Date:                       Thu, 16 Apr 2026   Prob (F-statistic):          2.85e-105
Time:                               09:25:21   Log-Likelihood:                -97121.
No. Observations:                      53458   AIC:                         1.943e+05
Df Residuals:                          53448   BIC:                         1.944e+05
Df Model:                                  9                                         
Covariance Type:                     cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
-

## Notes

1. The key DiD parameter is the coefficient on `treated:post_merger`.
2. Standard errors are clustered at the hospital (`ccn`) level.
3. Bankruptcy uses a linear probability model (OLS).